In [ ]:
import os
import pickle
from copy import deepcopy
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "model_data"
OUT_DIR = ROOT / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
def scatter_softmax(scores: torch.Tensor, index: torch.Tensor, num_nodes: int,
                    eps: float = 1e-16) -> torch.Tensor:
    if scores.numel() == 0:
        return scores
    max_per_node = scores.new_full((num_nodes,), float("-inf"))
    max_per_node = max_per_node.scatter_reduce(
        0, index, scores, reduce="amax", include_self=True
    )
    scores = scores - max_per_node[index]
    exp_scores = scores.exp()
    denom = torch.zeros(num_nodes, dtype=scores.dtype, device=scores.device)
    denom = denom.index_add(0, index, exp_scores)
    return exp_scores / (denom[index] + eps)


class AttentionBlock(nn.Module):
    def __init__(self, in_features, leaky_relu_slope=0.2, share_weights=False):
        super().__init__()
        self.linear_l = nn.Linear(in_features, in_features, bias=False)
        self.linear_r = (
            self.linear_l
            if share_weights
            else nn.Linear(in_features, in_features, bias=False)
        )
        self.attn = nn.Linear(in_features, 1, bias=False)
        self.activation = nn.LeakyReLU(leaky_relu_slope)

    def forward(self, x, src, dst, num_nodes):
        g_l = self.linear_l(x)
        g_r = self.linear_r(x)
        e = self.attn(self.activation(g_l[dst] + g_r[src])).squeeze(-1)
        return scatter_softmax(e, dst, num_nodes)


class MessagePassingLayer(nn.Module):
    def __init__(self, node_in, edge_dim, embedding_size, embedder_layers,
                 embedder_hidden, aggregation="sum", use_edges=True):
        super().__init__()
        if aggregation not in ("sum", "mean"):
            raise ValueError(f"Unknown aggregation {aggregation!r}")
        if embedder_layers < 1:
            raise ValueError("embedder_layers must be >= 1")

        self.node_in = node_in
        self.edge_dim = edge_dim if use_edges else 0
        self.embedding_size = embedding_size
        self.aggregation = aggregation
        self.use_edges = use_edges
        self.activation = nn.LeakyReLU()

        msg_in = node_in + self.edge_dim
        layers = []
        if embedder_layers == 1:
            layers.append(nn.Linear(msg_in, embedding_size))
        else:
            layers.append(nn.Linear(msg_in, embedder_hidden))
            for _ in range(embedder_layers - 2):
                layers.append(nn.Linear(embedder_hidden, embedder_hidden))
            layers.append(nn.Linear(embedder_hidden, embedding_size))
        self.embedders = nn.ModuleList(layers)

        self.attention = AttentionBlock(node_in)
        self.w = nn.Linear(node_in + embedding_size, node_in + embedding_size)

    def message(self, x, edges, src):
        m = torch.cat((x[src], edges), dim=1) if self.use_edges else x[src]
        for layer in self.embedders:
            m = self.activation(layer(m))
        return m

    def forward(self, x, adj, edges=None):
        if self.use_edges and edges is None:
            raise ValueError("Layer expects edge features but got edges=None")

        num_nodes = x.shape[0]
        src, dst = adj[0], adj[1]
        alpha = self.attention(x, src, dst, num_nodes)
        msg = alpha.unsqueeze(-1) * self.message(x, edges, src)

        agg = torch.zeros(
            num_nodes, self.embedding_size, dtype=msg.dtype, device=x.device
        )
        agg = agg.index_add(0, dst, msg)
        if self.aggregation == "mean":
            deg = torch.zeros(num_nodes, dtype=msg.dtype, device=x.device)
            deg = deg.index_add(0, dst, torch.ones_like(dst, dtype=msg.dtype))
            agg = agg / deg.clamp(min=1.0).unsqueeze(-1)

        combined = torch.cat((x, agg), dim=1)
        return self.activation(self.w(combined))


class GNNEncoder(nn.Module):
    def __init__(self, node_data_len, edge_dim, embedding_size, embedder_layers,
                 embedder_hidden, num_layers, aggregation="sum", use_edges=True,
                 no_norm=False):
        super().__init__()
        self.no_norm = no_norm
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(num_layers):
            node_in = node_data_len + i * embedding_size
            self.layers.append(MessagePassingLayer(
                node_in=node_in,
                edge_dim=edge_dim,
                embedding_size=embedding_size,
                embedder_layers=embedder_layers,
                embedder_hidden=embedder_hidden,
                aggregation=aggregation,
                use_edges=use_edges,
            ))
            self.norms.append(nn.LayerNorm(node_in + embedding_size))
        self.out_dim = node_data_len + num_layers * embedding_size

    def forward(self, x, adj, edges=None):
        
        for i, (layer, norm) in enumerate(zip(self.layers, self.norms)):

            x = layer(x, adj, edges)
            if not self.no_norm:
                x = norm(x)
                x = F.normalize(x, dim=1)
        return x



class Predictor(nn.Module):
    def __init__(self, in_size, layer_size, layer_num):
        super().__init__()
        self.input_layer = nn.Linear(in_size, layer_size)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(layer_size, layer_size) for _ in range(layer_num)]
        )
        self.output_layer = nn.Linear(layer_size, 1)
        self.activation = nn.ReLU()
        self.norms = nn.ModuleList(
            [nn.LayerNorm(layer_size) for _ in range(layer_num + 1)]
        )

    def forward(self, x):
        x = self.norms[0](self.activation(self.input_layer(x)))
        for i, layer in enumerate(self.hidden_layers):
            x = self.norms[i + 1](self.activation(layer(x)))
        return self.output_layer(x)


class RMSELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps

    def forward(self, pred, target):
        return torch.sqrt(self.mse(pred, target) + self.eps)


def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


print("model defined")

In [ ]:
@dataclass
class Config:
    node_data_len: int = 33
    edge_dim: int = 4
    seed: int = 29

    embedding_size: int = 3
    embedder_layers: int = 5
    embedder_hidden: int = 34
    num_layers: int = 1
    no_norm: bool = False
    aggregation: str = "sum"
    predictor_layer_size: int = 50
    predictor_layer_num: int = 4

    learning_rate: float = 7.628516574302889e-3
    weight_decay: float = 1e-5
    epochs: int = 200
    accumulation_steps: int = 16
    lr_step_size: int = 10
    lr_gamma: float = 0.99
    min_delta: float = 1e-5

    device: str = field(
        default_factory=lambda: "cuda:0" if torch.cuda.is_available() else "cpu"
    )

    use_wandb: bool = False


SWEEP_CONFIG = {
    "method": "random",
    "metric": {"name": "val_rmse", "goal": "minimize"},
    "parameters": {
        "learning_rate":
         {"distribution": "log_uniform_values", "min": 1e-7, "max": 1e-1},
        "batch_size": {
            "distribution": "q_log_uniform_values", "q": 2, "min": 1, "max": 40
        },
        "predictor_layer_size": {"values": [64, 96, 128]},
        "aggregation": {"values": ["mean", "sum"]},
        "embed_layer": {
            "values": [1,2]
        },
        "embed_size": {
            "values": [2,4,8,16]
        },
        "embedder_nn_layer_num": {
            "values": [1,2,3,4,5]
        },
        "embedder_nn_layer_size": {
            "distribution": "q_log_uniform_values", "q": 8, "min": 32, "max": 64
        },
    },
}


def config_from_wandb(wc, base):
    return replace(
        base,
        learning_rate=float(wc.learning_rate),
        accumulation_steps=int(wc.batch_size),
        num_layers=int(wc.embed_layer),
        embedding_size=int(wc.embed_size),
        embedder_layers=int(wc.embedder_nn_layer_num),
        embedder_hidden=int(wc.embedder_nn_layer_size),
        predictor_layer_size=int(wc.predictor_layer_size),
        predictor_layer_num=4,
        aggregation=str(wc.aggregation),
        use_wandb=True,
    )

def _wandb_log(cfg, data):
    if cfg.use_wandb:
        import wandb
        wandb.log(data)


def load_graph_splits(cfg):
    data_dir = DATA_DIR
    splits = {
        name: np.load(data_dir / f"{name}.npy", allow_pickle=True)
        for name in ("train", "val", "test")
    }
    print(
        f"[data] predefined split -> train={len(splits['train'])} "
        f"val={len(splits['val'])} test={len(splits['test'])}"
    )
    return splits["train"], splits["val"], splits["test"]


def load_heat_scaler(cfg):
    with open(DATA_DIR / "scalers.pkl", "rb") as f:
        scalers = pickle.load(f)
    print("[data] loaded train-only target scaler for physical-unit RMSE")
    return scalers["target_scaler"]


def unpack_unit(unit, cfg, device):
    node = torch.as_tensor(unit["nodes"], dtype=torch.float32).reshape(
        -1, cfg.node_data_len
    )
    if node.shape[0] <= 1:
        return None
    edge = torch.as_tensor(unit["edges"], dtype=torch.float32).reshape(
        -1, cfg.edge_dim
    )
    adj = torch.as_tensor(unit["adj"], dtype=torch.long).reshape(2, -1)
    y = torch.as_tensor(unit["y"], dtype=torch.float32).reshape(-1)
    return node.to(device), edge.to(device), adj.to(device), y.to(device)


def build_models(cfg):
    device = torch.device(cfg.device)
    model = GNNEncoder(
        node_data_len=cfg.node_data_len,
        edge_dim=cfg.edge_dim,
        embedding_size=cfg.embedding_size,
        embedder_layers=cfg.embedder_layers,
        embedder_hidden=cfg.embedder_hidden,
        num_layers=cfg.num_layers,
        aggregation=cfg.aggregation,
        use_edges=True,
        no_norm=cfg.no_norm,
    ).to(device)
    predictor = Predictor(
        in_size=model.out_dim,
        layer_size=cfg.predictor_layer_size,
        layer_num=cfg.predictor_layer_num,
    ).to(device)
    model.apply(init_weights)
    predictor.apply(init_weights)
    return model, predictor


@torch.no_grad()
def evaluate(model, predictor, loader, cfg, loss_func):
    device = torch.device(cfg.device)
    model.eval()
    predictor.eval()
    preds, targets = [], []
    for unit in loader:
        parsed = unpack_unit(unit, cfg, device)
        if parsed is None:
            continue
        node, edge, adj, y = parsed
        out = predictor(model(node, adj, edge)).flatten()
        preds.append(out.cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds)
    targets = torch.cat(targets)
    rmse = loss_func(preds, targets).item()
    r2 = r2_score(targets.numpy(), preds.numpy())
    return r2, rmse, preds.numpy(), targets.numpy()


print("configuration and data pipeline defined")


In [ ]:
def train(cfg):
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    device = torch.device(cfg.device)
    print(f"[setup] device = {device}")

    train_set, val_set, test_set = load_graph_splits(cfg)
    heat_scaler = load_heat_scaler(cfg)

    train_loader = DataLoader(train_set, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

    model, predictor = build_models(cfg)
    params = list(model.parameters()) + list(predictor.parameters())
    n_params = sum(p.numel() for p in params if p.requires_grad)
    print(f"[setup] trainable parameters = {n_params}")
    _wandb_log(cfg, {"parameter_count": n_params})

    loss_func = RMSELoss()
    optimizer = torch.optim.Adam(
        params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=cfg.lr_step_size, gamma=cfg.lr_gamma
    )

    train_curve, val_curve = [], []
    best_val = float("inf")
    best_state = None


    for epoch in range(cfg.epochs):
        model.train()
        predictor.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_losses = []
        seen_in_batch = 0

        for unit in train_loader:
            parsed = unpack_unit(unit, cfg, device)
            if parsed is None:
                continue
            node, edge, adj, y = parsed

            out = predictor(model(node, adj, edge)).flatten()
            loss = loss_func(out, y)
            (loss / cfg.accumulation_steps).backward()

            epoch_losses.append(loss.item())
            seen_in_batch += 1
            if seen_in_batch == cfg.accumulation_steps:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                seen_in_batch = 0

        if seen_in_batch > 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        scheduler.step()
        train_rmse = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
        val_r2, val_rmse, _, _ = evaluate(
            model, predictor, val_loader, cfg, loss_func
        )
        train_curve.append(train_rmse)
        val_curve.append(val_rmse)
        print(
            f"[epoch {epoch + 1:02d}/{cfg.epochs}] "
            f"lr={scheduler.get_last_lr()[0]:.2e} "
            f"train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f} "
            f"val_r2={val_r2:.4f}"
        )
        _wandb_log(cfg, {
            "epoch": epoch,
            "lr": scheduler.get_last_lr()[0],
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "val_r2": val_r2,
        })

        if val_rmse < best_val - cfg.min_delta:
            best_val = val_rmse
            best_state = {
                "model": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "predictor": {
                    k: v.detach().cpu().clone()
                    for k, v in predictor.state_dict().items()
                },
            }

        if epoch >=5 and val_r2<0.1:
            break

    if best_state is not None:
        model.load_state_dict(best_state["model"])
        predictor.load_state_dict(best_state["predictor"])

    post_training(
        cfg, model, predictor, loss_func, heat_scaler,
        train_loader, val_loader, test_loader, train_curve, val_curve
    )
    return model, predictor


def _physical_rmse(scaler, preds, targets):
    p = scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    t = scaler.inverse_transform(targets.reshape(-1, 1)).flatten()
    return float(np.sqrt(np.mean((p - t) ** 2)))


def post_training(cfg, model, predictor, loss_func, heat_scaler,
                  train_loader, val_loader, test_loader, train_curve, val_curve):
    print("\n[post] final evaluation on best checkpoint")
    test_r2 = test_rmse = test_rmse_phys = float("nan")
    for name, loader in (
        ("train", train_loader), ("val", val_loader), ("test", test_loader)
    ):
        r2, rmse, preds, targets = evaluate(
            model, predictor, loader, cfg, loss_func
        )
        phys = _physical_rmse(heat_scaler, preds, targets)
        print(f"  {name:5s}: R2={r2:.4f} RMSE={rmse:.4f} rmse(phys)={phys:.3f}")
        _wandb_log(cfg, {
            f"{name}_r2_final": r2,
            f"{name}_rmse_final": rmse,
            f"{name}_rmse_phys": phys,
        })
        if name == "test":
            test_r2, test_rmse, test_rmse_phys = r2, rmse, phys

    model_path = os.path.join(
        str(OUT_DIR), f"gnn_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    pred_path = os.path.join(
        str(OUT_DIR), f"mlp_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    torch.save(model.state_dict(), model_path)
    torch.save(predictor.state_dict(), pred_path)
    print(f"[post] saved model     -> {model_path}")
    print(f"[post] saved predictor -> {pred_path}")

    ckpt_path = os.path.join(
        str(OUT_DIR), f"checkpoint_R2_{test_r2:.4f}_RMSE_{test_rmse:.4f}.pt"
    )
    torch.save({
        "model_state": model.state_dict(),
        "predictor_state": predictor.state_dict(),
        "config": asdict(cfg),
        "metrics": {"test_r2": test_r2, "test_rmse": test_rmse, "test_rmse_phys": test_rmse_phys},
    }, ckpt_path)
    print(f"[post] saved full checkpoint -> {ckpt_path}")

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(train_curve) + 1), train_curve, label="train RMSE")
    plt.plot(range(1, len(val_curve) + 1), val_curve, label="val RMSE")
    plt.xlabel("epoch")
    plt.ylabel("RMSE")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(str(OUT_DIR), "loss_curve.png"), dpi=120)
    plt.show()


print("training pipeline defined")


In [ ]:
cfg = Config()
train(cfg)

In [ ]:
def train_sweep():
    import wandb
    wandb.init()
    cfg = config_from_wandb(wandb.config, Config(use_wandb=True))
    try:
        if cfg.use_wandb:
            wandb.config.update({f"cfg/{k}": v for k, v in asdict(cfg).items()}, allow_val_change=True)
        train(cfg)
    finally:
        wandb.finish()


def run_sweep(count=20, project="", entity=None):
    import wandb
    sweep_id = wandb.sweep(SWEEP_CONFIG, project=project, entity=entity)
    wandb.agent(sweep_id, function=train_sweep, count=count)



In [ ]:
# Run experiments on Wandb
import wandb
wandb.login()
run_sweep()

In [ ]:
def load_checkpoint(path, device=None, no_norm=None):
    ckpt = torch.load(path, map_location="cpu")
    if "config" not in ckpt or "model_state" not in ckpt:
        raise ValueError(
            "Expected a bundled checkpoint containing config, model_state, "
            "and predictor_state"
        )

    cfg = Config(**ckpt["config"])
    if no_norm is not None:
        cfg = replace(cfg, no_norm=no_norm)
    dev = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    cfg = replace(cfg, device=dev)

    model, predictor = build_models(cfg)
    model.load_state_dict(ckpt["model_state"], strict=not cfg.no_norm)
    predictor.load_state_dict(ckpt["predictor_state"])
    model.eval()
    predictor.eval()
    return model, predictor, cfg


def plot_regression(preds, targets, title, scaler=None, save_path=None):
    preds = np.asarray(preds).reshape(-1)
    targets = np.asarray(targets).reshape(-1)
    if scaler is not None:
        preds = scaler.inverse_transform(preds.reshape(-1, 1)).ravel()
        targets = scaler.inverse_transform(targets.reshape(-1, 1)).ravel()
        xlabel, ylabel = "Predicted (phys)", "Ground truth (phys)"
    else:
        xlabel, ylabel = "Predicted (scaled)", "Ground truth (scaled)"

    lo = float(min(preds.min(), targets.min()))
    hi = float(max(preds.max(), targets.max()))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(preds, targets, s=12, alpha=0.45)
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="green", label="x = y")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=120)
        print(f"[reproduce] saved plot -> {save_path}")
    plt.show()
    return fig


def reproduce_from_checkpoint(path, device=None, plot=True, physical=True,
                              save_plots=True, no_norm=False):
    model, predictor, cfg = load_checkpoint(path, device, no_norm=no_norm)
    train_set, val_set, test_set = load_graph_splits(cfg)
    heat_scaler = load_heat_scaler(cfg)
    loss_func = RMSELoss()

    loaders = {
        "train": DataLoader(train_set, batch_size=1),
        "val": DataLoader(val_set, batch_size=1),
        "test": DataLoader(test_set, batch_size=1),
    }

    print(f"[reproduce] loaded {path}")
    print(f"[reproduce] evaluating on device={cfg.device}")
    results = {}
    plot_dir = None
    if plot and save_plots:
        plot_dir = Path(str(OUT_DIR)) / "reproduce_plots"
        plot_dir.mkdir(parents=True, exist_ok=True)

    for name, loader in loaders.items():
        r2, rmse, preds, targets = evaluate(
            model, predictor, loader, cfg, loss_func
        )
        phys = _physical_rmse(heat_scaler, preds, targets)
        print(f"  {name:5s}: R2={r2:.4f} RMSE={rmse:.4f} rmse(phys)={phys:.3f}")
        results[name] = {
            "r2": r2,
            "rmse": rmse,
            "rmse_phys": phys,
            "preds": preds,
            "targets": targets,
        }

        if plot:
            unit = "phys" if physical else "scaled"
            title = f"{name}  R2={r2:.4f}  RMSE={rmse:.4f}"
            if physical:
                title += f"  RMSE(phys)={phys:.3f}"
            save_path = None
            if plot_dir is not None:
                save_path = str(plot_dir / f"{name}_regression_{unit}.png")
            plot_regression(
                preds,
                targets,
                title=title,
                scaler=heat_scaler if physical else None,
                save_path=save_path,
            )

    return results, model, predictor, cfg




In [ ]:
reproduce_from_checkpoint("./models/checkpoint_gubem__R2_0.9330_RMSE_0.2661.pt")